# Phase 1 verification + Phase 2 (mRMR)

Two jobs in one run:

**A. Verify NSGA-II in YOUR environment.** The published NSGA-II numbers were produced on
scikit-learn 1.8.0; yours is 1.6.1. Random Search already ran on yours. This re-runs NSGA-II
here so both methods share one environment. It is a **verification**, not a replacement: if
the numbers reproduce within tolerance we report independent confirmation and change nothing.

**B. Phase 2 — mRMR**, a non-evolutionary reference. This closes the paper's biggest
acknowledged gap: whether the instability is specific to NSGA-II or general.

Pre-registered, locked before any result is seen:
- mRMR uses `mrmr-selection` (FCQ variant: F-statistic + Pearson correlation, **not** classic
  mutual information — this must be stated in the paper).
- `k in {2, 4, 8}`, fixed in advance.
- Ranking computed **per bootstrap, from training data only** — never the whole dataset.
- Reported for the **data-resampling condition only**. mRMR is deterministic, so re-running it
  on fixed data returns the identical subset; Phi ~ 1 is an artifact and must not enter the
  main table.
- Same leakage controls, same OOB evaluation, same stability metrics as everything else.

Runtime ~90 minutes. Checkpointed after every (dataset, job) — re-running skips finished work.


## 1. Environment — run this first; it fails fast if anything is missing

In [ ]:
import sys, platform, json
print("Python:", sys.version.split()[0], "|", platform.system(), platform.release())

# If mrmr is missing:   pip install mrmr-selection
# (do NOT use pymrmr - it needs a C compiler and usually fails to build on Windows)

import numpy, scipy, sklearn, deap
ENV = {m.__name__: m.__version__ for m in (numpy, scipy, sklearn, deap)}
ENV["python"] = sys.version.split()[0]

MRMR_OK = True
try:
    from mrmr import mrmr_classif
    import mrmr as _m
    ENV["mrmr"] = getattr(_m, "__version__", "unknown")
    print("mrmr import OK ->", ENV["mrmr"])
except Exception as e:
    MRMR_OK = False
    print("!! mrmr NOT available:", e)
    print("!! Part B will be SKIPPED. Part A (NSGA-II verification) still runs.")
    print("!! To fix: pip install mrmr-selection   then restart the kernel.")

for k, v in ENV.items():
    print(f"  {k:>10}: {v}")

## 2. CONFIG

In [ ]:
from pathlib import Path
PROJ_ROOT = Path.cwd()
if not (PROJ_ROOT/"src").exists() and (PROJ_ROOT.parent/"src").exists():
    PROJ_ROOT = PROJ_ROOT.parent
SRC_DIR, DATA_DIR, RESULTS_DIR = PROJ_ROOT/"src", PROJ_ROOT/"data", PROJ_ROOT/"results"
RESULTS_DIR.mkdir(exist_ok=True)

VERIFY_FILE = RESULTS_DIR/"phase1_nsga_verification.json"
MRMR_FILE   = RESULTS_DIR/"phase2_mrmr.json"

DATASETS = ["breast_cancer", "colon", "leukemia"]
N_RUNS   = 10
POP_SIZE, N_GEN = 40, 20
MRMR_KS  = [2, 4, 8]

assert SRC_DIR.exists(),  f"src/ not found under {PROJ_ROOT}"
assert DATA_DIR.exists(), f"data/ not found under {PROJ_ROOT}"
print("PROJ_ROOT:", PROJ_ROOT)

## 3. Imports from `src/` — nothing reimplemented

In [ ]:
sys.path.insert(0, str(SRC_DIR))
import numpy as np, time
import data as dm, nested_cv as ncv, ga as ga_mod, bootstrap as bs, stability as stab
from stability_ci import nogueira_stability
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
print("ok")

## 4. Shared helpers (identical to the Phase 1 notebook)

In [ ]:
def balanced_acc(yt, yp):
    cs = np.unique(yt)
    if len(cs) < 2: return float((yp == yt).mean())
    return float(np.mean([(yp[yt == c] == c).mean() for c in cs]))

def oob_accuracy(X, y, bidx, features):
    oob = np.setdiff1d(np.arange(len(y)), np.unique(bidx))
    if len(oob) < 5 or len(np.unique(y[oob])) < 2 or len(features) == 0:
        return None, len(oob)
    Xtr, ytr = X[np.ix_(bidx, features)], y[bidx]
    Xte, yte = X[np.ix_(oob, features)], y[oob]
    sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
    clf = KNeighborsClassifier(n_neighbors=min(5, len(ytr))); clf.fit(Xtr, ytr)
    return balanced_acc(yte, clf.predict(Xte)), len(oob)

def draws_for(y, n=N_RUNS):
    """Reproduces EXACTLY the bootstrap draws used by every earlier experiment."""
    rng = np.random.default_rng(0)
    return [bs.stratified_bootstrap_indices(y, rng) for _ in range(n)]

## 5. PART A — verify NSGA-II in this environment

Same seeds (0-9), same config, same draws as the published runs.


In [ ]:
ver = json.load(open(VERIFY_FILE)) if VERIFY_FILE.exists() else {}
for ds in DATASETS:
    if ds in ver:
        print(f"SKIP {ds} (done)"); continue
    X, y, tag = dm.load(ds); p = X.shape[1]
    draws = draws_for(y)
    runs, t0 = [], time.time()
    for i in range(N_RUNS):
        bidx = draws[i]
        Xb, yb = X[bidx], y[bidx]
        cv = ncv.make_cv(len(yb), seed=i, groups=bidx)
        n_ev = {"n": 0}
        def ef(mask, Xb=Xb, yb=yb, cv=cv, g=bidx, c=n_ev):
            c["n"] += 1
            return ncv.eval_mask(Xb, yb, mask, cv, groups=g)
        front = ga_mod.run_nsga2(p, ef, pop_size=POP_SIZE, n_gen=N_GEN, seed=i)
        front = [(m, a, k) for (m, a, k) in front if k > 0]
        knee = stab.knee_point_mask(front)
        feats = np.where(knee)[0].tolist() if knee is not None else []
        acc, on = oob_accuracy(X, y, bidx, feats)
        runs.append({"seed": i, "n_evals": n_ev["n"], "front_size": len(front),
                     "knee_features": feats, "knee_n_features": len(feats),
                     "oob_accuracy": acc, "oob_set_size": int(on),
                     "front": [{"n_sel": int(k), "acc": float(a),
                                "features": np.where(m)[0].tolist()} for m, a, k in front]})
        print(f"  [{ds}] run {i+1}/{N_RUNS} front={len(front)} k={len(feats)} "
              f"oob={acc if acc is None else round(acc,3)} [{time.time()-t0:.0f}s]", flush=True)
    ver[ds] = {"env": ENV, "runs": runs, "p": p}
    json.dump(ver, open(VERIFY_FILE, "w"), indent=2)
    print(f"--- CHECKPOINT {ds} saved ---", flush=True)
print("PART A DONE ->", VERIFY_FILE)

### A2. Did it reproduce? (verification, not replacement)

Tolerance is scaled to the size of the held-out set: with ~22 OOB samples on Colon a single
differently-classified case moves accuracy by ~4.5 points, so a fixed 0.002 threshold was
never appropriate there. We use **2 / OOB_n**, i.e. roughly two samples' worth.


In [ ]:
published_oob = {"breast_cancer": 0.930, "colon": 0.680, "leukemia": 0.774}
published_phi = {"breast_cancer": 0.3665, "colon": 0.0063, "leukemia": 0.0086}

print(f"{'dataset':>14} {'OOB pub':>8} {'OOB here':>9} {'tol':>7} {'ok':>4} | "
      f"{'Phi pub':>8} {'Phi here':>9}")
for ds in DATASETS:
    runs = ver[ds]["runs"]; p = ver[ds]["p"]
    accs = [r["oob_accuracy"] for r in runs if r["oob_accuracy"] is not None]
    oob_n = float(np.median([r["oob_set_size"] for r in runs]))
    tol = 2.0/oob_n
    got = float(np.median(accs))
    Z = []
    for r in runs:
        u = np.zeros(p, bool)
        for s in r["front"]: u[s["features"]] = True
        Z.append(u)
    phi, _ = nogueira_stability(np.array(Z))
    ok = abs(got - published_oob[ds]) <= tol
    print(f"{ds:>14} {published_oob[ds]:>8.3f} {got:>9.3f} {tol:>7.3f} {str(ok):>4} | "
          f"{published_phi[ds]:>8.4f} {phi:>9.4f}")
print()
print("Interpretation is pre-registered:")
print("  all within tolerance -> report independent confirmation; change no published number")
print("  a material difference -> bring the numbers back and we decide together, with data")

## 6. PART B — mRMR (Phase 2)

Skipped automatically if the import failed.


In [ ]:
if not MRMR_OK:
    print("SKIPPED - mrmr not installed. Send Part A results; we can do Part B separately.")
else:
    import pandas as pd
    mr = json.load(open(MRMR_FILE)) if MRMR_FILE.exists() else {}
    for ds in DATASETS:
        if ds in mr:
            print(f"SKIP {ds} (done)"); continue
        X, y, tag = dm.load(ds); p = X.shape[1]
        draws = draws_for(y)
        runs, t0 = [], time.time()
        for i in range(N_RUNS):
            bidx = draws[i]
            # ranking from TRAINING DATA ONLY, never the full dataset
            Xtr = pd.DataFrame(X[bidx]); ytr = pd.Series(y[bidx])
            sel = {}
            ranked = mrmr_classif(X=Xtr, y=ytr, K=max(MRMR_KS), show_progress=False)
            for k in MRMR_KS:
                feats = [int(f) for f in ranked[:k]]
                acc, on = oob_accuracy(X, y, bidx, feats)
                sel[str(k)] = {"features": feats, "oob_accuracy": acc, "oob_set_size": int(on)}
            runs.append({"bootstrap_id": i, "by_k": sel})
            print(f"  [{ds}] mRMR run {i+1}/{N_RUNS} "
                  f"k=8 oob={sel['8']['oob_accuracy'] if sel['8']['oob_accuracy'] is None else round(sel['8']['oob_accuracy'],3)} "
                  f"[{time.time()-t0:.0f}s]", flush=True)
        mr[ds] = {"env": ENV, "ks": MRMR_KS, "runs": runs, "p": p}
        json.dump(mr, open(MRMR_FILE, "w"), indent=2)
        print(f"--- CHECKPOINT {ds} saved ---", flush=True)
    print("PART B DONE ->", MRMR_FILE)

### B2. mRMR stability — the number that answers the paper's biggest open question

In [ ]:
if MRMR_OK:
    print(f"{'dataset':>14} {'k':>3} {'Phi':>9} {'OOB med':>9} {'NSGA-II Phi':>12}")
    for ds in DATASETS:
        p = mr[ds]["p"]
        for k in MRMR_KS:
            Z, accs = [], []
            for r in mr[ds]["runs"]:
                u = np.zeros(p, bool); u[r["by_k"][str(k)]["features"]] = True; Z.append(u)
                a = r["by_k"][str(k)]["oob_accuracy"]
                if a is not None: accs.append(a)
            phi, _ = nogueira_stability(np.array(Z))
            print(f"{ds:>14} {k:>3} {phi:>9.4f} {np.median(accs):>9.3f} {published_phi[ds]:>12.4f}")
    print()
    print("mRMR is deterministic: on fixed data it returns the identical subset, so it has no")
    print("fixed-data stochasticity. That goes in one sentence of text, NOT in the main table.")

## 7. What to send back

- `results/phase1_nsga_verification.json`
- `results/phase2_mrmr.json`  (if Part B ran)
- the console output of sections 5, A2, 6 and B2

If mrmr would not install, send Part A anyway — it is the more important half.
